# 🎮 꼬맨틀 게임

## 게임 소개
임베딩 기반 **코사인 유사도**로 단어의 의미적 유사도를 계산하는 게임입니다.

### 📋 게임 규칙
1. `gpt-5.4-mini`가 랜덤하게 정답 단어를 생성합니다.
2. **VS Code 상단 입력창**에 단어를 입력하여 추측합니다.
3. **임베딩 벡터의 코사인 유사도**로 정답과의 유사도(0-100)를 계산합니다.
4. '포기'를 입력하면 정답을 알려줍니다.
5. 정답을 맞출 때까지 계속 도전합니다.

### 💡 입력 방법
- 셀을 실행하면 **VS Code 화면 상단 중앙**에 입력창이 자동으로 나타납니다.
- 입력창에 단어를 입력하고 Enter를 누르세요.
- '기록' 또는 'ㄱ' 입력으로 시도 기록 확인이 가능합니다.

### 🔬 기술 스택
- **LLM 모델**: Azure OpenAI `gpt-5.4-mini`
- **임베딩 모델**: Azure OpenAI `text-embedding-3-large`
- **접속 방식**: APIM gateway + APIM 구독 키
- **SDK**: OpenAI Python SDK `AzureOpenAI`
- **유사도 계산**: 코사인 유사도 (Cosine Similarity)
- **수식**: $\cos(\theta) = \frac{A \cdot B}{||A|| \times ||B||}$
- **스케일링**: Semantle 방식 (-1~1 → 0~100 선형 변환)
  - 공식: `((cosine + 1) / 2) * 100`
- **게임 언어**: 한글 단어

### 💡 게임 팁
- **한글 단어**로 입력하세요.
- 카테고리 힌트를 활용하여 범위를 좁혀보세요.
- 상대적 점수 변화에 집중하세요.
- 90점대가 나오면 정답이 아주 가깝습니다.

## 1. 환경 설정

In [18]:
import warnings
warnings.filterwarnings('ignore')

import json
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from dotenv import load_dotenv
from openai import AzureOpenAI

# 환경 변수 로드
dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")
api_key = os.getenv("AZURE_OPENAI_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
chat_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.4-mini")
embedding_deployment = os.getenv("EMBEDDING_MODEL_NAME", "text-embedding-3-large")

if not azure_endpoint or not api_key:
    raise ValueError(".env 파일에 AZURE_OPENAI_ENDPOINT와 AZURE_OPENAI_KEY를 설정하세요.")

# APIM gateway를 통해 Chat Completions와 Embeddings를 모두 호출합니다.
openai_client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=azure_endpoint,
    default_headers={"Ocp-Apim-Subscription-Key": api_key},
)

print("✅ Azure OpenAI 클라이언트 초기화 완료")
print(f"LLM deployment: {chat_deployment}")
print(f"Embedding deployment: {embedding_deployment}")
print(f"API version: {api_version}")

✅ Azure OpenAI 클라이언트 초기화 완료
LLM deployment: gpt-5.4-mini
Embedding deployment: text-embedding-3-large
API version: 2025-04-01-preview


## 2. 임베딩 기반 꼬맨틀 게임 클래스

In [19]:
import random

class LLMKomantleGame:
    """LLM을 활용한 꼬맨틀 게임"""

    ANSWER_CATEGORIES = {
        "동물": ["호랑이", "참새", "거북이", "사자", "독수리", "개미", "고양이", "강아지"],
        "과일": ["포도", "수박", "망고", "사과", "바나나", "딸기", "오렌지"],
        "음식": ["밥", "스파게티", "만두", "김치", "피자", "치킨", "초밥"],
        "직업": ["간호사", "요리사", "경찰", "의사", "선생님", "프로그래머"],
        "장소": ["도서관", "해변", "광장", "학교", "병원", "공원", "카페"],
        "자연": ["구름", "계곡", "무지개", "바다", "산", "하늘", "강", "별"],
        "감정": ["기쁨", "흥분", "외로움", "행복", "슬픔", "사랑", "분노"],
        "사물": ["시계", "의자", "가방", "책", "컴퓨터", "자동차", "신발"],
    }
    recent_answers_by_category = {category: [] for category in ANSWER_CATEGORIES}

    def __init__(self, client):
        self.client = client
        self.answer = None
        self.answer_vec = None  # 정답 임베딩 캐싱용
        self.attempts = []
        self.attempt_count = 0

    def _build_answer_prompt(self, category: str, forbidden_words: List[str], variation_seed: int) -> str:
        forbidden_text = ", ".join(forbidden_words) if forbidden_words else "없음"
        return f"""한국어 명사 중에서 꼬맨틀 게임의 정답으로 적합한 단어 하나를 선택해주세요.
일상적인 명사가 좋습니다.

이번 게임의 카테고리: {category}
다양성 번호: {variation_seed}

아래 단어들은 예시이거나 최근에 이미 나온 단어입니다. 정답으로 절대 선택하지 마세요.
금지 단어: {forbidden_text}

선택 기준:
- 반드시 이번 게임의 카테고리에 속하는 한글 명사 하나만 선택하세요.
- 금지 단어와 너무 비슷한 단어는 피하세요.
- 너무 전문적인 단어, 고유명사, 외래어, 복합 명사보다는 일상적인 단어를 고르세요.
- 매번 다른 단어가 나오도록 다양성 번호를 참고하세요.

응답은 반드시 JSON 객체 하나로만 답변하세요.

JSON 형식:
{{
  "answer": "선택한 한글 단어",
  "category": "{category}"
}}
"""

    async def generate_answer(self, max_retries: int = 3) -> Tuple[str, str]:
        """LLM이 정답 단어를 생성 (실패 시 재시도)"""
        category_names = list(self.ANSWER_CATEGORIES.keys())

        for attempt in range(max_retries):
            selected_category = random.choice(category_names)
            recent_answers = self.recent_answers_by_category.get(selected_category, [])[-6:]
            forbidden_words = sorted(set(self.ANSWER_CATEGORIES[selected_category] + recent_answers))
            variation_seed = random.randint(100000, 999999)
            prompt = self._build_answer_prompt(selected_category, forbidden_words, variation_seed)

            try:
                response = self.client.chat.completions.create(
                    model=chat_deployment,
                    messages=[
                        {"role": "system", "content": "You return only valid JSON."},
                        {"role": "user", "content": prompt},
                    ],
                    response_format={"type": "json_object"},
                    temperature=1.2,
                    max_completion_tokens=200,
                )

                content = response.choices[0].message.content
                data = json.loads(content.strip())
                answer = data["answer"].strip()

                if answer in forbidden_words:
                    raise ValueError(f"금지 단어가 선택되었습니다: {answer}")

                self.answer = answer
                category = selected_category

                self.recent_answers_by_category.setdefault(category, []).append(self.answer)
                self.recent_answers_by_category[category] = self.recent_answers_by_category[category][-10:]

                # 정답 임베딩 미리 계산 및 캐싱 (성능 최적화)
                self.answer_vec = self.get_embedding(self.answer)

                print(f"✅ 정답 생성 성공 (시도 {attempt + 1}/{max_retries})")
                return self.answer, category

            except Exception as e:
                print(f"⚠️ 정답 생성 실패 (시도 {attempt + 1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    print("🔄 재시도 중...")
                else:
                    print("❌ 최대 재시도 횟수 초과")
                    raise Exception(f"정답 생성 실패: {max_retries}번 시도 후에도 실패했습니다.")

    def get_embedding(self, text: str, max_retries: int = 2) -> np.ndarray:
        """텍스트를 벡터로 변환 (재시도 로직 포함)"""
        for attempt in range(max_retries):
            try:
                response = self.client.embeddings.create(
                    input=text,
                    model=embedding_deployment,
                )
                return np.array(response.data[0].embedding)
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"⚠️ 임베딩 생성 실패, 재시도 중... ({attempt + 1}/{max_retries})")
                    continue
                else:
                    raise Exception(f"임베딩 생성 실패: {e}")

    def calculate_cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """코사인 유사도 계산 (0~100 스케일)

        Semantle 방식: 코사인 유사도 -1~1을 0~100으로 선형 매핑
        - 이점: 모델 독립적, 일관된 감각, 투명한 해석
        - 공식: ((cosine + 1) / 2) * 100
        """
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)

        if norm1 == 0 or norm2 == 0:
            return 0.0

        # 코사인 유사도 계산 (-1 ~ 1)
        cosine_sim = float(dot_product / (norm1 * norm2))
        cosine_sim = max(-1.0, min(1.0, cosine_sim))  # 클리핑

        # Semantle 방식: -1~1 → 0~100 선형 변환
        similarity = ((cosine_sim + 1.0) / 2.0) * 100.0

        return float(similarity)

    async def calculate_similarity(self, word: str) -> Tuple[float, str]:
        """임베딩 기반 코사인 유사도 계산"""
        if word == self.answer:
            return 100.0, "🎉 정답입니다!"

        # 캐싱된 정답 벡터 사용 (성능 최적화)
        word_vec = self.get_embedding(word)

        # 코사인 유사도 계산
        similarity = self.calculate_cosine_similarity(self.answer_vec, word_vec)

        # 재조정된 유사도 구간 (더 엄격하고 세분화)
        if similarity >= 95:
            reason = "거의 정답 수준"
        elif similarity >= 90:
            reason = "매우 밀접한 의미적 관계"
        elif similarity >= 85:
            reason = "강한 의미적 연관성"
        elif similarity >= 75:
            reason = "어느 정도 의미적 관련성"
        elif similarity >= 60:
            reason = "약한 의미적 관련성"
        elif similarity >= 40:
            reason = "거의 관련 없음"
        else:
            reason = "의미적 연관성 매우 낮음"

        return similarity, reason

    def get_feedback(self, similarity: float) -> str:
        """유사도에 따른 피드백 (재조정된 구간)"""
        if similarity >= 95:
            return "🔥🔥🔥 완전 불타오른다! 바로 코앞이에요!"
        elif similarity >= 90:
            return "🔥🔥 불타오른다! 거의 다 왔어요!"
        elif similarity >= 85:
            return "🔥 뜨겁다! 꽤 가깝습니다!"
        elif similarity >= 75:
            return "😊 좋아요! 어느 정도 관련이 있어요"
        elif similarity >= 60:
            return "🤔 음... 조금 관련이 있는 것 같아요"
        elif similarity >= 40:
            return "😕 아직 멀어요"
        else:
            return "❄️ 매우 멀어요. 완전 다른 분야네요!"

    async def guess(self, word: str) -> Dict:
        """단어 추측"""
        self.attempt_count += 1

        if word == "포기":
            return {
                "type": "give_up",
                "answer": self.answer,
                "attempts": self.attempt_count - 1
            }

        if word == self.answer:
            self.attempts.append({
                "word": word,
                "similarity": 100.0,
                "reason": "정답",
                "rank": self.attempt_count,
            })
            return {
                "type": "correct",
                "attempts": self.attempt_count,
                "word": word,
                "similarity": 100.0,
            }

        similarity, reason = await self.calculate_similarity(word)
        feedback = self.get_feedback(similarity)

        self.attempts.append({
            "word": word,
            "similarity": similarity,
            "reason": reason,
            "rank": self.attempt_count
        })

        return {
            "type": "guess",
            "word": word,
            "similarity": similarity,
            "reason": reason,
            "feedback": feedback,
            "attempts": self.attempt_count
        }

    def show_history(self, top_n: int = 10):
        """시도 기록 보기"""
        if not self.attempts:
            print("아직 시도한 단어가 없습니다.")
            return

        sorted_attempts = sorted(self.attempts, key=lambda x: x['similarity'], reverse=True)

        print(f"\n📊 상위 {min(top_n, len(sorted_attempts))}개 시도:")
        print("=" * 80)
        print(f"{'순위':<4} {'단어':<12} {'유사도':<8} {'이유':<50}")
        print("=" * 80)

        for i, attempt in enumerate(sorted_attempts[:top_n], 1):
            similarity = attempt['similarity']
            bar_length = int(similarity / 5)
            bar = "█" * bar_length
            reason = attempt['reason'][:45] + "..." if len(attempt['reason']) > 45 else attempt['reason']
            print(f"{i:2d}.  {attempt['word']:<12} {similarity:5.1f} {bar:<20} {reason}")

        print("=" * 80)
        print(f"총 시도 횟수: {self.attempt_count}\n")

print("✅ 꼬맨틀 게임 클래스 정의 완료")

✅ 꼬맨틀 게임 클래스 정의 완료


## 3. 게임 플레이

In [20]:
async def play_game():
    """꼬맨틀 게임 플레이"""
    game = LLMKomantleGame(openai_client)

    print("🎲 LLM이 정답 단어를 생성하는 중...\n")
    answer, category = await game.generate_answer()

    print("=" * 80)
    print("🎮 게임 시작!")
    print(f"📁 카테고리: {category}")
    print(f"💡 정답은 {len(answer)}글자 한글 단어입니다.")
    print("\n명령어:")
    print("  - 한글 단어 입력: 추측하기 (예: 고양이, 강아지, 사과)")
    print("  - '포기': 정답 보기 및 게임 종료")
    print("  - '기록' 또는 'ㄱ': 시도 기록 보기")
    print("=" * 80)

    game_over = False

    while not game_over:
        print("\n")
        word = input("💬 한글 단어를 입력하세요: ").strip()

        if not word:
            print("⚠️ 한글 단어를 입력해주세요!")
            continue

        if word in ["기록", "ㄱ"]:
            game.show_history(top_n=15)
            continue

        print(f"\n🎯 입력: '{word}'")
        print("⏳ 임베딩으로 유사도를 계산하는 중...\n")

        result = await game.guess(word)

        if result["type"] == "give_up":
            print("\n😢 포기하셨습니다.")
            print("=" * 60)
            print(f"📝 정답: {result['answer']}")
            print(f"🎯 시도 횟수: {result['attempts']}번")
            print("=" * 60)
            game.show_history()
            game_over = True

        elif result["type"] == "correct":
            print("\n🎉🎉🎉 축하합니다! 정답을 맞추셨습니다! 🎉🎉🎉")
            print("=" * 60)
            print(f"✅ 정답: {result['word']}")
            print(f"🎯 시도 횟수: {result['attempts']}번")
            print("=" * 60)
            game.show_history()
            game_over = True

        else:
            print("─" * 80)
            print(f"📊 유사도: {result['similarity']:.1f}점")
            print(f"💭 평가: {result['reason']}")
            print(f"🎭 {result['feedback']}")
            print(f"📈 시도 횟수: {result['attempts']}번")
            print("─" * 80)

    if game.attempts:
        avg_similarity = sum(a["similarity"] for a in game.attempts) / len(game.attempts)
        max_similarity = max(a["similarity"] for a in game.attempts)
        best_guess = max(game.attempts, key=lambda x: x["similarity"])

        print("\n📈 게임 통계")
        print("=" * 60)
        print(f"🎯 정답: {game.answer}")
        print(f"📊 총 시도 횟수: {game.attempt_count}")
        print(f"📈 평균 유사도: {avg_similarity:.2f}")
        print(f"🏆 최고 유사도: {max_similarity:.2f}")
        print(f"👍 가장 가까웠던 추측: '{best_guess['word']}' ({best_guess['similarity']:.1f}점)")
        print("=" * 60)

# 게임 시작
await play_game()

🎲 LLM이 정답 단어를 생성하는 중...

✅ 정답 생성 성공 (시도 1/3)
🎮 게임 시작!
📁 카테고리: 음식
💡 정답은 2글자 한글 단어입니다.

명령어:
  - 한글 단어 입력: 추측하기 (예: 고양이, 강아지, 사과)
  - '포기': 정답 보기 및 게임 종료
  - '기록' 또는 'ㄱ': 시도 기록 보기



🎯 입력: '냉면'
⏳ 임베딩으로 유사도를 계산하는 중...

────────────────────────────────────────────────────────────────────────────────
📊 유사도: 70.6점
💭 평가: 약한 의미적 관련성
🎭 🤔 음... 조금 관련이 있는 것 같아요
📈 시도 횟수: 1번
────────────────────────────────────────────────────────────────────────────────



🎯 입력: '초밥'
⏳ 임베딩으로 유사도를 계산하는 중...

────────────────────────────────────────────────────────────────────────────────
📊 유사도: 74.7점
💭 평가: 약한 의미적 관련성
🎭 🤔 음... 조금 관련이 있는 것 같아요
📈 시도 횟수: 2번
────────────────────────────────────────────────────────────────────────────────



🎯 입력: '생선'
⏳ 임베딩으로 유사도를 계산하는 중...

────────────────────────────────────────────────────────────────────────────────
📊 유사도: 68.1점
💭 평가: 약한 의미적 관련성
🎭 🤔 음... 조금 관련이 있는 것 같아요
📈 시도 횟수: 3번
────────────────────────────────────────────────────────────────────────────────



🎯 입력: '치킨'
⏳ 임베딩으로